In [ ]:
import sys
from pathlib import Path

from truststore import inject_into_ssl

inject_into_ssl()

sys.path.append(str(Path.cwd().parent))

import requests  # noqa: E402
from src.config.settings import IndexerSettings, SnowSettings  # noqa: E402

In [ ]:
indexer_settings = IndexerSettings()
snow_settings = SnowSettings()
print(f"indexer_settings: {indexer_settings}")
print(f"snow_settings: {snow_settings}")

# proxies
HTTP_PROXY = indexer_settings.http_proxy
HTTPS_PROXY = indexer_settings.https_proxy
NO_PROXY = indexer_settings.no_proxy

# snow settings
snow_url = snow_settings.servicenow_url
snow_client_id = snow_settings.servicenow_client_id
# secret accessable through snow_settings.servicenow_client_secret

print(snow_settings.token_url)

In [ ]:
token_url = snow_settings.token_url
session = requests.Session()
data = {
    "grant_type": "client_credentials",
    "client_id": snow_client_id,
    "client_secret": snow_settings.servicenow_client_secret.strip() if snow_settings.servicenow_client_secret else "",
    "scope": snow_settings.servicenow_oauth_scope,
}
headers = {"Content-Type": "application/x-www-form-urlencoded"}
print("Requesting ServiceNow access token")
resp = session.post(token_url, data=data, headers=headers, verify=True, proxies={"http": HTTP_PROXY, "https": HTTPS_PROXY})
print(resp.status_code)

try:
    print(resp.json())
except ValueError:
    print(resp.text)
resp.raise_for_status()
token = resp.json().get("access_token")
if not token:
    raise RuntimeError("ServiceNow OAuth response did not contain an access_token")
print(token)

In [ ]:
def get_category(article):
    meta_description = article.get("meta_description", "")
    if "eakte-nutzer*innen" in meta_description['value'].lower():
        return "user"
    elif "eakte-fachadministrator*innen" in meta_description['value'].lower():
        return "admin"
    else:
        return "general"

In [ ]:
session.headers.update({"Authorization": f"Bearer {token}"})
META_FIELDS = ",".join([
    "kb_category", "kb_knowledge_base", "author", "workflow_state",
    "sys_created_on", "sys_updated_on", "valid_to",
    "sys_view_count", "keywords", "meta_description",
])

params = {
    "limit": snow_settings.servicenow_page_size,
    "fields": META_FIELDS,          # <-- add this (KM API param, not sysparm_fields)
}
data = session.get(snow_url, params=params, verify=True,
                   proxies={"http": HTTP_PROXY, "https": HTTPS_PROXY})
articles = data.json()["result"]["articles"]

for article in articles:
    print(article["number"], article["id"], list(article.keys()))
    # articles are split in generally two scpoes: for users and for admins. this is defined in the meta_description field
    # this is important for the vectordb, because we should be able to distinguish between the two scopes when searching for articles. 
    # this enables us to provide more specific search results based on the user's role or needs.
    scope = get_category(article.get("fields", {}))
    print(article.get("fields"))     # <-- this is the check that matters

In [ ]:
fields = ",".join([
    "sys_id", "number", "short_description", "text",
    "kb_knowledge_base", "kb_category", "topic", "category",
    "workflow_state", "published", "valid_to",
    "sys_created_on", "sys_updated_on", "author",
    "sys_view_count", "meta_description", "keywords",
])

article_url = "https://lhm.service-now.com/api/sn_km_api/knowledge/articles/{}"
full_articles = []

for article in articles:
    r = session.get(
        article_url.format(article['id'].split(':')[1]),
        params={
            "sysparm_fields": fields,
            "sysparm_display_value": "all",   # resolve reference fields
        },
        verify=True, proxies={"http": HTTP_PROXY, "https": HTTPS_PROXY},
    )
    r.raise_for_status()
    full_articles.append(r.json()["result"])

print(len(full_articles))


In [ ]:
print(full_articles[0].keys())

In [ ]:
from src.loaders.snow_loader import SnowLoader  # noqa: E402

loader = SnowLoader(config=snow_settings)

docs = loader.load_documents()


In [ ]:
from pprint import pprint  # noqa: E402

pprint(docs[0].metadata)
print(len(docs))